# 04 — Segmentación ML

Agrupamos a los clientes con **K-means** y calculamos la afinidad entre segmentos y categorías
con **DuckDB**. Utilizamos los perfiles de Gold para identificar hábitos de compra y orientar
la asignación de incentivos.

**Criterios del modelo:**

- Utilizamos recencia, frecuencia mensual, gasto promedio y franja preferida; transformamos la franja en columnas numéricas y estandarizamos las variables.
- Reservamos la categoría preferida para interpretar los grupos.
- Entrenamos con variables observables, sin las etiquetas ocultas de la generación sintética.
- Definimos cuatro segmentos y una semilla fija para reproducir el escenario.
- Calculamos la afinidad con las compras de los dos primeros meses, incluidos los canjes, porque también reflejan elección de categorías.
- Consideramos afinidad alta cuando `indice_afinidad > 1.2`.

Incorporamos el segmento al perfil de cada cliente y resumimos las afinidades en una tabla de Gold.


In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "doc.md").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Ejecutar desde la raíz del proyecto o notebooks/.")
SILVER = PROJECT_ROOT / "data" / "silver"
GOLD = PROJECT_ROOT / "data" / "gold"

SEED = 22434
NUM_SEGMENTOS = 4
UMBRAL_AFINIDAD = 1.2
FRANJAS = ["mañana", "almuerzo", "tarde", "noche"]
SEGMENTOS = ["Leales", "Madrugadores", "En riesgo", "Ocasionales"]
VARIABLES_NUMERICAS = ["recencia_dias", "frecuencia_mensual", "gasto_promedio"]
# El bloque one-hot de franja debe pesar como una sola variable (ver sección de preparación).
PESO_FRANJA = 1 / np.sqrt(len(FRANJAS))


def sql_text(value):
    return "'" + str(value).replace("'", "''") + "'"


connection = duckdb.connect(":memory:")
connection.execute("SET threads = 1")
for table in ("clientes", "productos", "compras", "lineas_compra"):
    parquet = sql_text(SILVER / f"{table}.parquet")
    connection.execute(f"CREATE VIEW {table} AS SELECT * FROM read_parquet({parquet})")

# Al reejecutar, perfil_clientes ya trae un segmento previo: se descarta para recalcularlo.
perfil_clientes = connection.execute(
    f"SELECT * FROM read_parquet({sql_text(GOLD / 'perfil_clientes.parquet')}) ORDER BY cliente_id"
).df().drop(columns="segmento", errors="ignore")
print(f"DuckDB {duckdb.__version__} — scikit-learn {sklearn.__version__}")
print(f"perfil_clientes: {len(perfil_clientes)} clientes")
perfil_clientes.head()

DuckDB 1.5.5 — scikit-learn 1.9.1
perfil_clientes: 1000 clientes


,cliente_id,recencia_dias,frecuencia_mensual,gasto_promedio,franja_preferida,categoria_preferida
0,1,17,1.0,23.75,almuerzo,acompañamientos
1,2,2,14.0,48.80,mañana,desayunos
2,3,1,18.5,99.88,noche,hamburguesas
3,4,2,2.0,21.98,almuerzo,acompañamientos
4,5,6,1.0,57.65,tarde,bebidas


## Preparación de variables

K-means utiliza distancias entre variables numéricas. Preparamos los datos en tres pasos:

1. Convertimos `franja_preferida` en cuatro columnas 0/1, una por franja.
2. Estandarizamos las columnas para que las diferencias de escala no dominen la agrupación.
3. Ponderamos el bloque de franja por `1/√4 = 0.5`. Así, sus cuatro columnas aportan en conjunto una unidad de varianza, comparable con cada variable numérica.

Esta preparación equilibra la contribución de recencia, frecuencia, gasto y horario.


In [2]:
preprocesamiento = ColumnTransformer(
    [
        ("numericas", StandardScaler(), VARIABLES_NUMERICAS),
        ("franja", Pipeline([
            ("one_hot", OneHotEncoder(categories=[FRANJAS], sparse_output=False)),
            ("escala", StandardScaler()),
        ]), ["franja_preferida"]),
    ],
    transformer_weights={"numericas": 1.0, "franja": PESO_FRANJA},
)
X = preprocesamiento.fit_transform(perfil_clientes)
columnas_modelo = list(preprocesamiento.get_feature_names_out())

assert not any("categoria" in columna for columna in columnas_modelo), "categoria_preferida no debe entrar al modelo"
assert X.shape == (len(perfil_clientes), len(VARIABLES_NUMERICAS) + len(FRANJAS))
assert not np.isnan(X).any(), "La matriz del modelo no debe tener nulos"

varianza = pd.Series(X.var(axis=0), index=columnas_modelo).round(3)
print("Varianza por columna del modelo:")
print(varianza.to_string())
print(f"Varianza total del bloque franja: {varianza.filter(like='franja').sum():.3f} (≈ una variable)")

Varianza por columna del modelo:
numericas__recencia_dias             1.00
numericas__frecuencia_mensual        1.00
numericas__gasto_promedio            1.00
franja__franja_preferida_mañana      0.25
franja__franja_preferida_almuerzo    0.25
franja__franja_preferida_tarde       0.25
franja__franja_preferida_noche       0.25
Varianza total del bloque franja: 1.000 (≈ una variable)


## Diagnóstico del número de grupos

Planteamos cuatro segmentos por su utilidad para interpretar el comportamiento de compra.
Evaluamos la inercia y el coeficiente de silueta para k = 2…8. La silueta varía entre −1 y 1;
los valores mayores indican grupos más compactos y separados.

Con k = 4 obtenemos una silueta de 0.4280, mientras k = 6 alcanza 0.4499.
Mantenemos cuatro grupos por la interpretación de negocio, aunque esta elección
no maximiza la silueta.


In [3]:
diagnostico = []
for k in range(2, 9):
    modelo_k = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(X)
    diagnostico.append({
        "k": k,
        "inercia": round(modelo_k.inertia_, 1),
        "silueta": round(silhouette_score(X, modelo_k.labels_), 4),
    })
diagnostico = pd.DataFrame(diagnostico).set_index("k")
diagnostico

,inercia,silueta
k,,
2,2562.6,0.3380
3,1768.0,0.3867
4,1265.2,0.4280
5,1122.1,0.4298
6,982.5,0.4499
7,878.6,0.4343
8,796.1,0.4482


## Entrenamiento de K-means

Entrenamos cuatro grupos con 20 inicializaciones y seleccionamos la solución de menor inercia.
Evaluamos la estabilidad con otras diez semillas y comparamos las particiones mediante
el índice de Rand ajustado (ARI). Un valor de 1 indica los mismos grupos, aunque cambie
su numeración.


In [4]:
kmeans = KMeans(n_clusters=NUM_SEGMENTOS, n_init=20, random_state=SEED)
grupos = kmeans.fit_predict(X)

estabilidad = [
    adjusted_rand_score(grupos, KMeans(n_clusters=NUM_SEGMENTOS, n_init=20, random_state=semilla).fit_predict(X))
    for semilla in range(1, 11)
]
silueta = silhouette_score(X, grupos)

assert len(set(grupos)) == NUM_SEGMENTOS, "K-means debe producir cuatro grupos no vacíos"
assert min(estabilidad) > 0.9, f"Segmentación inestable entre semillas (ARI mínimo {min(estabilidad):.3f})"
print(f"Silueta con k={NUM_SEGMENTOS}: {silueta:.4f}")
print(f"ARI frente a otras 10 semillas: mínimo {min(estabilidad):.4f}, promedio {np.mean(estabilidad):.4f}")
print("Clientes por grupo:", dict(sorted(pd.Series(grupos).value_counts().items())))

Silueta con k=4: 0.4280
ARI frente a otras 10 semillas: mínimo 1.0000, promedio 1.0000
Clientes por grupo: {0: 250, 1: 255, 2: 249, 3: 246}


## Interpretación de los grupos

K-means asigna números a los grupos. Para interpretarlos, resumimos su tamaño, recencia,
frecuencia, gasto, distribución de la franja preferida y categoría preferida más común.
Expresamos los indicadores en sus unidades originales; la categoría apoya la interpretación
y no participa en el entrenamiento.


In [5]:
perfil_grupos = perfil_clientes.assign(grupo=grupos)

resumen_grupos = perfil_grupos.groupby("grupo").agg(
    clientes=("cliente_id", "size"),
    recencia_dias=("recencia_dias", "mean"),
    frecuencia_mensual=("frecuencia_mensual", "mean"),
    gasto_promedio=("gasto_promedio", "mean"),
)
pct_franja = (
    pd.crosstab(perfil_grupos["grupo"], perfil_grupos["franja_preferida"], normalize="index")
    .reindex(columns=FRANJAS, fill_value=0)
    .add_prefix("pct_")
)
categoria_comun = perfil_grupos.groupby("grupo")["categoria_preferida"].agg(
    lambda categorias: categorias.value_counts().index[0]
).rename("categoria_mas_comun")

resumen_grupos = resumen_grupos.join(pct_franja).join(categoria_comun).round(2)
resumen_grupos

,clientes,recencia_dias,frecuencia_mensual,gasto_promedio,pct_mañana,pct_almuerzo,pct_tarde,pct_noche,categoria_mas_comun
grupo,,,,,,,,,
0,250,1.94,12.00,102.68,0.31,0.28,0.17,0.24,hamburguesas
1,255,26.08,6.38,58.78,0.04,0.65,0.04,0.27,ensaladas
2,249,2.79,8.69,39.12,1.00,0.00,0.00,0.00,desayunos
3,246,11.25,1.36,26.41,0.17,0.26,0.42,0.15,acompañamientos


### Asignación de nombres

Asignamos nombres mediante reglas explícitas sobre las características de cada grupo.
Seguimos este orden para mantener una interpretación reproducible:

| Orden | Segmento | Regla | Interpretación |
|---|---|---|---|
| 1 | Ocasionales | Menor `frecuencia_mensual` | Presentan pocas compras; su recencia también puede ser alta |
| 2 | Madrugadores | Mayor proporción de franja `mañana` entre los restantes | Concentran su actividad en la mañana |
| 3 | En riesgo | Mayor `recencia_dias` entre los dos restantes | Presentan frecuencia intermedia y menor actividad reciente |
| 4 | Leales | Grupo restante | Presentan mayor frecuencia y gasto |

Verificamos que los indicadores de cada segmento coincidan con esta interpretación.
Las comprobaciones detectan si un cambio en los datos produce grupos que requieren
una interpretación distinta.


In [6]:
def nombrar_grupos(resumen):
    pendientes = set(resumen.index)

    def elegir(columna, criterio):
        valores = resumen.loc[sorted(pendientes), columna]
        grupo = valores.idxmin() if criterio == "min" else valores.idxmax()
        pendientes.remove(grupo)
        return grupo

    nombres = {}
    nombres[elegir("frecuencia_mensual", "min")] = "Ocasionales"
    nombres[elegir("pct_mañana", "max")] = "Madrugadores"
    nombres[elegir("recencia_dias", "max")] = "En riesgo"
    nombres[pendientes.pop()] = "Leales"
    return {int(grupo): nombre for grupo, nombre in nombres.items()}


nombres_grupo = nombrar_grupos(resumen_grupos)
resumen_segmentos = resumen_grupos.rename(index=nombres_grupo).rename_axis("segmento").loc[SEGMENTOS]

assert sorted(nombres_grupo.values()) == sorted(SEGMENTOS), "Cada segmento debe asignarse a un solo grupo"
leales, madrugadores, riesgo, ocasionales = (resumen_segmentos.loc[s] for s in SEGMENTOS)
assert leales["frecuencia_mensual"] == resumen_segmentos["frecuencia_mensual"].max(), "Leales: deben ser los más frecuentes"
assert leales["gasto_promedio"] == resumen_segmentos["gasto_promedio"].max(), "Leales: deben tener el mayor gasto"
assert madrugadores["pct_mañana"] >= 0.8, "Madrugadores: la mayoría debe preferir la mañana"
assert riesgo["recencia_dias"] > max(leales["recencia_dias"], madrugadores["recencia_dias"]), "En riesgo: menos recientes que los activos"
assert riesgo["frecuencia_mensual"] > ocasionales["frecuencia_mensual"], "En riesgo: compraban más que los Ocasionales"

perfil_clientes["segmento"] = pd.Series(grupos).map(nombres_grupo).to_numpy()
print("Grupo → segmento:", dict(sorted(nombres_grupo.items())))
resumen_segmentos

Grupo → segmento: {0: 'Leales', 1: 'En riesgo', 2: 'Madrugadores', 3: 'Ocasionales'}


,clientes,recencia_dias,frecuencia_mensual,gasto_promedio,pct_mañana,pct_almuerzo,pct_tarde,pct_noche,categoria_mas_comun
segmento,,,,,,,,,
Leales,250,1.94,12.00,102.68,0.31,0.28,0.17,0.24,hamburguesas
Madrugadores,249,2.79,8.69,39.12,1.00,0.00,0.00,0.00,desayunos
En riesgo,255,26.08,6.38,58.78,0.04,0.65,0.04,0.27,ensaladas
Ocasionales,246,11.25,1.36,26.41,0.17,0.26,0.42,0.15,acompañamientos


Examinamos el porcentaje de clientes de cada segmento que prefiere cada categoría.
Esta distribución complementa la interpretación: los Madrugadores, por ejemplo,
concentran sus preferencias en desayunos y bebidas.


In [7]:
(
    pd.crosstab(perfil_clientes["segmento"], perfil_clientes["categoria_preferida"], normalize="index")
    .loc[SEGMENTOS]
    .mul(100)
    .round(1)
)

categoria_preferida,acompañamientos,bebidas,desayunos,ensaladas,hamburguesas,pollo,postres
segmento,,,,,,,
Leales,0.0,0.0,0.0,0.0,69.6,26.0,4.4
Madrugadores,0.0,19.7,80.3,0.0,0.0,0.0,0.0
En riesgo,2.7,2.4,0.4,78.4,12.9,3.1,0.0
Ocasionales,49.2,42.7,1.2,0.8,1.6,0.8,3.7


## `afinidad_segmento_categoria`

Calculamos una fila por segmento y categoría con las compras de los dos primeros meses:

- `pct_compras_segmento`: porcentaje de compras del segmento que incluyen al menos un producto de la categoría.
- `pct_compras_total`: porcentaje equivalente sobre todas las compras de la ventana.
- `indice_afinidad`: cociente entre ambos porcentajes. Un valor mayor que 1 indica una presencia de la categoría superior al promedio.

Calculamos el índice antes de redondear e incluimos las 28 combinaciones de cuatro segmentos
y siete categorías.


In [8]:
inicio_mes1, fin_ventana = connection.execute("""
    SELECT date_trunc('month', min(fecha_hora)) AS inicio_mes1,
           date_trunc('month', min(fecha_hora)) + INTERVAL 2 MONTH AS fin_ventana
    FROM compras
""").fetchone()
print(f"Ventana: [{inicio_mes1.date()}, {fin_ventana.date()})")

segmentos_sql = "[" + ", ".join(sql_text(segmento) for segmento in SEGMENTOS) + "]"
connection.register("segmentos_df", perfil_clientes[["cliente_id", "segmento"]])

afinidad_segmento_categoria = connection.execute(f"""
    WITH compras_ventana AS (
        SELECT c.id AS compra_id, s.segmento
        FROM compras c
        JOIN segmentos_df s ON s.cliente_id = c.cliente_id
        WHERE c.fecha_hora >= {sql_text(inicio_mes1)}::TIMESTAMP
          AND c.fecha_hora < {sql_text(fin_ventana)}::TIMESTAMP
    ),
    compra_categoria AS (
        SELECT DISTINCT cv.compra_id, cv.segmento, p.categoria
        FROM compras_ventana cv
        JOIN lineas_compra l ON l.compra_id = cv.compra_id
        JOIN productos p ON p.id = l.producto_id
    ),
    compras_segmento AS (
        SELECT segmento, count(*)::DOUBLE AS compras FROM compras_ventana GROUP BY 1
    ),
    compras_total AS (
        SELECT count(*)::DOUBLE AS compras FROM compras_ventana
    ),
    categorias AS (
        SELECT DISTINCT categoria FROM productos
    ),
    categoria_segmento AS (
        SELECT segmento, categoria, count(*) AS compras FROM compra_categoria GROUP BY 1, 2
    ),
    categoria_total AS (
        SELECT categoria, count(*) AS compras FROM compra_categoria GROUP BY 1
    ),
    proporciones AS (
        SELECT
            cs.segmento,
            cat.categoria,
            coalesce(sc.compras, 0) / cs.compras AS prop_segmento,
            coalesce(ct.compras, 0) / t.compras AS prop_total
        FROM compras_segmento cs
        CROSS JOIN categorias cat
        CROSS JOIN compras_total t
        LEFT JOIN categoria_segmento sc ON sc.segmento = cs.segmento AND sc.categoria = cat.categoria
        LEFT JOIN categoria_total ct ON ct.categoria = cat.categoria
    )
    SELECT
        segmento,
        categoria,
        round(100 * prop_segmento, 2) AS pct_compras_segmento,
        round(100 * prop_total, 2) AS pct_compras_total,
        round(prop_segmento / prop_total, 4) AS indice_afinidad
    FROM proporciones
    ORDER BY list_position({segmentos_sql}, segmento), categoria
""").df()
afinidad_segmento_categoria.head(7)

Ventana: [2025-01-01, 2025-03-01)


,segmento,categoria,pct_compras_segmento,pct_compras_total,indice_afinidad
0,Leales,acompañamientos,31.75,21.22,1.4965
1,Leales,bebidas,23.12,32.44,0.7125
2,Leales,desayunos,14.02,31.22,0.4490
3,Leales,ensaladas,8.52,18.72,0.4550
4,Leales,hamburguesas,64.62,38.04,1.6984
5,Leales,pollo,56.40,32.30,1.7463
6,Leales,postres,43.02,24.86,1.7300


Comparamos la afinidad de cada segmento con el umbral de 1.2 y la relacionamos con
las categorías de los productos rezagados. Este cruce permite identificar a qué clientes
dirigir los incentivos de cada producto.


In [9]:
matriz_afinidad = afinidad_segmento_categoria.pivot(index="segmento", columns="categoria", values="indice_afinidad").loc[SEGMENTOS]
print(matriz_afinidad.round(2).to_string())

ventas_productos = connection.execute(
    f"SELECT * FROM read_parquet({sql_text(GOLD / 'ventas_productos.parquet')})"
).df()
rezagados = ventas_productos.loc[ventas_productos["es_rezagado"]].merge(
    connection.execute("SELECT id AS producto_id, nombre FROM productos").df(), on="producto_id"
)
print("\nProductos rezagados y segmentos con afinidad > umbral en su categoría:")
for producto in rezagados.itertuples():
    afines = afinidad_segmento_categoria[
        (afinidad_segmento_categoria["categoria"] == producto.categoria)
        & (afinidad_segmento_categoria["indice_afinidad"] > UMBRAL_AFINIDAD)
    ]
    destino = ", ".join(f"{fila.segmento} ({fila.indice_afinidad:.2f})" for fila in afines.itertuples()) or "ninguno → todos los clientes"
    print(f"  {producto.nombre} [{producto.categoria}]: {destino}")

categoria     acompañamientos  bebidas  desayunos  ensaladas  hamburguesas  pollo  postres
segmento                                                                                  
Leales                   1.50     0.71       0.45       0.46          1.70   1.75     1.73
Madrugadores             0.26     1.65       2.26       0.30          0.16   0.12     0.24
En riesgo                0.89     0.58       0.49       3.11          0.98   0.95     0.72
Ocasionales              1.88     1.43       0.26       0.12          0.28   0.20     0.77

Productos rezagados y segmentos con afinidad > umbral en su categoría:
  Hamburguesa especial [hamburguesas]: Leales (1.70)
  Sándwich de pollo especial [pollo]: Leales (1.75)
  Pie de manzana [postres]: Leales (1.73)
  Ensalada especial [ensaladas]: En riesgo (3.11)


## Validación y exportación

Comprobamos que cada cliente tenga un único segmento válido y que la tabla de afinidad
contenga una fila por segmento y categoría, sin valores nulos ni negativos.

También verificamos que el porcentaje total de compras por categoría coincida con
el promedio de los porcentajes de cada segmento ponderado por su número de compras,
con una tolerancia para el redondeo.

Guardamos en Gold el perfil con su segmento y la tabla de afinidad.


In [10]:
num_clientes = connection.execute("SELECT count(*) FROM clientes").fetchone()[0]
num_categorias = connection.execute("SELECT count(DISTINCT categoria) FROM productos").fetchone()[0]

assert len(perfil_clientes) == num_clientes, "Cada cliente debe tener una fila en perfil_clientes"
assert perfil_clientes["segmento"].isin(SEGMENTOS).all(), "Segmento fuera de los cuatro válidos"
assert perfil_clientes["segmento"].nunique() == NUM_SEGMENTOS, "Los cuatro segmentos deben tener clientes"
assert perfil_clientes.isna().sum().sum() == 0, "perfil_clientes no debe tener nulos"

assert len(afinidad_segmento_categoria) == NUM_SEGMENTOS * num_categorias, "Una fila por segmento y categoría"
assert not afinidad_segmento_categoria.duplicated(["segmento", "categoria"]).any()
assert afinidad_segmento_categoria.notna().all().all(), "afinidad_segmento_categoria no debe tener nulos"
assert (afinidad_segmento_categoria["indice_afinidad"] >= 0).all()

compras_por_segmento = connection.execute(f"""
    SELECT s.segmento, count(*) AS compras
    FROM compras c JOIN segmentos_df s ON s.cliente_id = c.cliente_id
    WHERE c.fecha_hora >= {sql_text(inicio_mes1)}::TIMESTAMP AND c.fecha_hora < {sql_text(fin_ventana)}::TIMESTAMP
    GROUP BY 1
""").df().set_index("segmento")["compras"]
peso_segmento = afinidad_segmento_categoria["segmento"].map(compras_por_segmento / compras_por_segmento.sum())
consistencia = (
    afinidad_segmento_categoria
    .assign(ponderado=afinidad_segmento_categoria["pct_compras_segmento"] * peso_segmento)
    .groupby("categoria")
    .agg(ponderado=("ponderado", "sum"), total=("pct_compras_total", "first"))
)
assert np.allclose(consistencia["ponderado"], consistencia["total"], atol=0.05), "pct_compras_total inconsistente con los segmentos"

columnas_perfil = ["cliente_id", *VARIABLES_NUMERICAS, "franja_preferida", "categoria_preferida", "segmento"]
GOLD.mkdir(parents=True, exist_ok=True)
connection.register("perfil_clientes_df", perfil_clientes[columnas_perfil])
connection.register("afinidad_df", afinidad_segmento_categoria)
connection.execute(f"COPY (SELECT * FROM perfil_clientes_df ORDER BY cliente_id) TO {sql_text(GOLD / 'perfil_clientes.parquet')} (FORMAT PARQUET, COMPRESSION ZSTD)")
connection.execute(f"COPY (SELECT * FROM afinidad_df ORDER BY list_position({segmentos_sql}, segmento), categoria) TO {sql_text(GOLD / 'afinidad_segmento_categoria.parquet')} (FORMAT PARQUET, COMPRESSION ZSTD)")

for name, esperado in (("perfil_clientes", num_clientes), ("afinidad_segmento_categoria", NUM_SEGMENTOS * num_categorias)):
    filas = connection.execute(f"SELECT count(*) FROM read_parquet({sql_text(GOLD / f'{name}.parquet')})").fetchone()[0]
    assert filas == esperado, f"{name}: se exportaron {filas} filas, se esperaban {esperado}"
    print(f"{name}: {filas} filas exportadas a Gold.")

sin_segmento = connection.execute(
    f"SELECT count(*) FROM read_parquet({sql_text(GOLD / 'perfil_clientes.parquet')}) WHERE segmento IS NULL"
).fetchone()[0]
assert sin_segmento == 0, "Hay clientes sin segmento en el Parquet exportado"
print("Clientes por segmento:")
print(perfil_clientes["segmento"].value_counts().reindex(SEGMENTOS).to_string())

connection.close()

perfil_clientes: 1000 filas exportadas a Gold.
afinidad_segmento_categoria: 28 filas exportadas a Gold.
Clientes por segmento:
segmento
Leales          250
Madrugadores    249
En riesgo       255
Ocasionales     246


## Resultados

Identificamos 250 Leales, 249 Madrugadores, 255 En riesgo y 246 Ocasionales.
Obtenemos una silueta de 0.4280 y un ARI mínimo de 1.0000 entre las diez semillas evaluadas.
Estos indicadores muestran la separación y la estabilidad de los grupos en el escenario sintético.

Relacionamos los segmentos con las categorías que compran con mayor frecuencia relativa.
Junto con las ventas por producto, esta información orienta las reglas de puntos.
La agrupación describe hábitos de consumo; no estima una probabilidad de abandono.
El modelo y su preprocesamiento permanecen en memoria, mientras los perfiles y afinidades
se almacenan como tablas.
